In [1]:
!pip install geemap
!pip install geetools


  Using cached Deprecated-1.2.18-py2.py3-none-any.whl.metadata (5.7 kB)
  Using cached xee-0.0.20-py3-none-any.whl.metadata (6.2 kB)
  Using cached yamlable-1.1.1-py2.py3-none-any.whl.metadata (3.1 kB)
  Using cached affine-2.4.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached cloudpickle-3.1.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached toolz-1.0.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached locket-1.0.0-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached zict-3.0.0-py2.py3-none-any.whl.metadata (899 bytes)
Using cached xee-0.0.20-py3-none-any.whl (31 kB)
Using cached affine-2.4.0-py3-none-any.whl (15 kB)
Using cached Deprecated-1.2.18-py2.py3-none-any.whl (10.0 kB)
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
    ------------------

In [1]:
import sys, subprocess
print(sys.executable) 
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U",
    "python-dotenv", "pydrive2", "oauth2client"
])

C:\Users\Dell\anaconda3\envs\vft-gee\python.exe


0

In [2]:
#Import Libraries
import ee 
import geemap
import geetools

In [4]:
#authenticate and initialize ee

ee.Authenticate()
#ee.Authenticate(auth_mode='notebook', cloud_api=True)


True

In [48]:
ee.Initialize()

In [51]:
# Define individual features
B1 = ee.Feature(ee.Geometry.Point([-79.21025, 35.14636]), {'name': 'B1', 'type': 'site'})
B2 = ee.Feature(ee.Geometry.Point([-79.19908, 35.14625]), {'name': 'B2', 'type': 'site'})
CM = ee.Feature(ee.Geometry.Point([-76.97426, 34.76602]), {'name': 'CM', 'type': 'site'})
CH = ee.Feature(ee.Geometry.Point([-76.87395833, 34.767175]), {'name': 'CH', 'type': 'site'})
ME = ee.Feature(ee.Geometry.Point([-77.47046, 34.64371]), {'name': 'ME', 'type': 'site'})
IA = ee.Feature(ee.Geometry.Point([-77.30881, 34.59154]), {'name': 'IA', 'type': 'site'})
GSP_BI = ee.Feature(ee.Geometry.Point([-78.30196, 34.08546]), {'name': 'GSP_BI', 'type': 'site'})
GSP_LI = ee.Feature(ee.Geometry.Point([-78.29995, 34.081985]), {'name': 'GSP_LI', 'type': 'site'})



In [52]:
# Create a FeatureCollection
VFT_sites = ee.FeatureCollection([B1, B2, CM,CH,ME,IA,GSP_BI,GSP_LI])
#bands = ['SR_B1','SR_B3','SR_B4','SR_B5', 'SR_B7','QA_PIXEL'] 
bands= ['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5', 'SR_B6','SR_B7','QA_PIXEL']
#bands = image.bandNames()         # add desirable bands, and QA_pixel (cloud info). Depends on the index to calculate. Modify this when switching satellites
roi_GSP = ee.Geometry.Rectangle([-78.33, 34.05, -78.24, 34.13])  # [minLon, minLat, maxLon, maxLat]
roi_Lej = ee.Geometry.Rectangle([-77.55, 34.52, -77.23, 34.73])  # [minLon, minLat, maxLon, maxLat]
roi_Cro = ee.Geometry.Rectangle([-77.02, 34.72, -76.83, 34.82])  # [minLon, minLat, maxLon, maxLat]
roi_BR = ee.Geometry.Rectangle([-79.225, 35.13, -79.19, 35.16])  # [minLon, minLat, maxLon, maxLat]



In [60]:


collection = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')  # LANDSAT/LT05/C02/T1_L2    LANDSAT/LC08/C02/T1_L2
    .filterDate('2024-12-01', '2025-8-25')   
    .filterBounds(roi_Lej)  # Change ROI as needed 
    .select(bands)  # Select required bands, make su;re QA_pixel is in there
)

# Check number of images remaining
print("Number of images after filtering:", collection.size().getInfo())

Number of images after filtering: 27


In [54]:
import time


In [61]:
def export_image(image):
    img_id = image.get('system:index').getInfo()
    
    # Convert to float and clip to roi
    image = image.float().clip(roi_Lej)  # modify this when switching areas

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=f'Landsat_{img_id}',
        folder='Landsat8_Lej',  # Google Drive Folder   # modify this when switching areas
        fileNamePrefix=f'Landsat_{img_id}',
        scale=30,  # 30m resolution
        region=roi_Lej.getInfo()['coordinates'],     # modify this when switching areas
        maxPixels=1e13
    )
    task.start()
    return task, img_id  # Return task and image ID for monitoring

# Convert collection to list and check if it contains images
num_images = collection.size().getInfo()
if num_images == 0:
    raise ValueError("🚨 No images found! ")

print(f"✅ Found {num_images} images. Starting export...")

# Export each image
image_list = collection.toList(num_images)
tasks = []

for i in range(num_images):
    img = ee.Image(image_list.get(i))
    task, img_id = export_image(img)
    tasks.append((task, img_id))  # Store task for monitoring

print("\n🚀 Batch export started. Monitoring progress...\n")

# Function to check task status
def check_task_status(tasks):
    while any(task.status()['state'] in ['READY', 'RUNNING'] for task, _ in tasks):
        print("\n🔄 Current Export Status:")
        for task, img_id in tasks:
            status = task.status()
            print(f"  - {img_id}: {status['state']}")
        time.sleep(100)  # Wait 5 minutes before checking again

    print("\n✅ All tasks completed!")

# Monitor progress
check_task_status(tasks)

✅ Found 27 images. Starting export...

🚀 Batch export started. Monitoring progress...


🔄 Current Export Status:
  - LC08_014036_20241208: RUNNING
  - LC08_014036_20250109: RUNNING
  - LC08_014036_20250125: RUNNING
  - LC08_014036_20250226: READY
  - LC08_014036_20250330: READY
  - LC08_014036_20250415: READY
  - LC08_014036_20250501: READY
  - LC08_014036_20250517: READY
  - LC08_014036_20250602: READY
  - LC08_014036_20250618: READY
  - LC08_014036_20250704: READY
  - LC08_014036_20250720: READY
  - LC08_014036_20250805: READY
  - LC08_015036_20241215: READY
  - LC08_015036_20241231: READY
  - LC08_015036_20250116: READY
  - LC08_015036_20250201: READY
  - LC08_015036_20250217: READY
  - LC08_015036_20250321: READY
  - LC08_015036_20250406: READY
  - LC08_015036_20250422: READY
  - LC08_015036_20250508: READY
  - LC08_015036_20250524: READY
  - LC08_015036_20250609: READY
  - LC08_015036_20250625: READY
  - LC08_015036_20250711: READY
  - LC08_015036_20250727: READY

🔄 Current Export

In [21]:
import os, glob
print(os.getcwd())

F:\VFT\VFT_github\zyao78VFTcode


In [45]:
from googleapiclient.errors import HttpError

def merge_folders_by_name(folder_name: str, delete_duplicates=True, dry_run=False):
    # Drive v2 query: use title (not name)
    q = ("title = '{0}' and "
         "mimeType = 'application/vnd.google-apps.folder' and trashed = false").format(folder_name)

    folders = drive.ListFile({
        'q': q,
        'supportsAllDrives': True,
        'includeItemsFromAllDrives': True,
        'maxResults': 1000
    }).GetList()

    if len(folders) < 2:
        print(f"Found {len(folders)} folder(s) named '{folder_name}'. Nothing to merge.")
        return

    target = folders[0]
    target_id = target['id']
    print(f"Found {len(folders)} folder(s) named '{folder_name}'.")
    print(f"Target: {target['title']} ({target_id})")

    svc = drive.auth.service

    for dup in folders[1:]:
        dup_id = dup['id']
        print(f"Merging: {dup['title']} ({dup_id}) -> {folder_name}")

        children = drive.ListFile({
            'q': f"'{dup_id}' in parents and trashed = false",
            'supportsAllDrives': True,
            'includeItemsFromAllDrives': True,
            'maxResults': 1000
        }).GetList()

        for it in children:
            file_id = it['id']
            title   = it['title']

            if dry_run:
                print(f"  would move: {title} -> target ({target_id})")
                continue

            # remove other parents and add target
            old_parents = ",".join(p['id'] for p in it.get('parents', []) if p['id'] != target_id)
            svc.files().update(
                fileId=file_id,
                addParents=target_id,
                removeParents=old_parents,
                supportsAllDrives=True
            ).execute()
            print(f"  moved: {title}")

        if not dry_run and delete_duplicates:
            svc.files().delete(fileId=dup_id, supportsAllDrives=True).execute()
            print(f"  deleted folder: {dup['title']} ({dup_id})")

    print("Done.")

In [22]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

# Sign in with your Google account (opens a browser tab)
gauth = GoogleAuth()
gauth.LocalWebserverAuth()
drive = GoogleDrive(gauth)

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?client_id=548363215141-24o3chdgv280l66mi2s58ebv2bga5kkm.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&access_type=online&response_type=code

Authentication successful.


In [39]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

# assume you've already authenticated: gauth = GoogleAuth(); gauth.LocalWebserverAuth()
# drive = GoogleDrive(gauth)

def _list(q: str):
    return drive.ListFile({
        'q': q,
        'supportsAllDrives': True,
        'includeItemsFromAllDrives': True
    }).GetList()



In [40]:


# Dry run
merge_folders_by_name("Landsat8_BR", dry_run=True)



Found 29 folder(s) named 'Landsat8_BR'.
Target: Landsat8_BR (19fgXJUseqMq3qA9oln_BVT93vf8TSe8N)
Merging: Landsat8_BR (1hAihZsoRwcrRMEdpGKeHvLoxpMuzkuNm)
  would move: Landsat_LC08_016035_20241206.tif -> Landsat8_BR
Merging: Landsat8_BR (1GmKJpeHLkuaWygO1QgtbufpshC23OcJD)
  would move: Landsat_LC08_016035_20250107.tif -> Landsat8_BR
Merging: Landsat8_BR (1oKENsNLUpuQ5RfrRCsgmyig78b2e5ua9)
  would move: Landsat_LC08_016035_20250123.tif -> Landsat8_BR
Merging: Landsat8_BR (1LqBFTJizRscJrVX956PjksCUcAWFcWfS)
  would move: Landsat_LC08_016035_20250224.tif -> Landsat8_BR
Merging: Landsat8_BR (1h8KcGT5QLtkV4hpitniycEIvdzmQ_kQL)
  would move: Landsat_LC08_016035_20250312.tif -> Landsat8_BR
Merging: Landsat8_BR (1gjomPxEne401T_ybHieY6Xl-eOyvW5IA)
  would move: Landsat_LC08_016035_20250328.tif -> Landsat8_BR
Merging: Landsat8_BR (1G7ZaNu0yD4YRe4P37kScR8cIAsj00REn)
  would move: Landsat_LC08_016035_20250413.tif -> Landsat8_BR
Merging: Landsat8_BR (1k3BqCi_igKxjOoY7OyYC7a0h79hxQJB8)
  would move: 

In [62]:
merge_folders_by_name("Landsat8_Lej", dry_run=False)


Found 27 folder(s) named 'Landsat8_Lej'.
Target: Landsat8_Lej (1EhSyJHAriyDjMPnMC-pjFS0xMPD4BBXa)
Merging: Landsat8_Lej (1ZueIojFnDHiQD73OV6VrhBtA5MjOwfZ3) -> Landsat8_Lej
  moved: Landsat_LC08_015036_20250711.tif
  deleted folder: Landsat8_Lej (1ZueIojFnDHiQD73OV6VrhBtA5MjOwfZ3)
Merging: Landsat8_Lej (1CmLwVvdUfLrrX0iXEwuEbwDn8rH-QLmX) -> Landsat8_Lej
  moved: Landsat_LC08_015036_20250625.tif
  deleted folder: Landsat8_Lej (1CmLwVvdUfLrrX0iXEwuEbwDn8rH-QLmX)
Merging: Landsat8_Lej (1ycjp3g_MRPEyMs2Ps2d2mxKtxDmJLqJF) -> Landsat8_Lej
  moved: Landsat_LC08_015036_20250609.tif
  deleted folder: Landsat8_Lej (1ycjp3g_MRPEyMs2Ps2d2mxKtxDmJLqJF)
Merging: Landsat8_Lej (130lm3Z_V457QrM28flbxr11ZH0wMzi8n) -> Landsat8_Lej
  moved: Landsat_LC08_015036_20250524.tif
  deleted folder: Landsat8_Lej (130lm3Z_V457QrM28flbxr11ZH0wMzi8n)
Merging: Landsat8_Lej (1-IZ7BAAf9n_hEE24irIUiTNujwTGn36I) -> Landsat8_Lej
  moved: Landsat_LC08_015036_20250508.tif
  deleted folder: Landsat8_Lej (1-IZ7BAAf9n_hEE24irIUi